# Moving Average Crossover Backtest (Quickstart)
- Data: yfinance (1d bars)
- Strategy: SMA fast vs slow
- Metrics: CAGR, Sharpe, MaxDD, Win rate


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

symbol = 'SPY'
df = yf.download(symbol, start='2015-01-01', progress=False)
df = df[['Close']].rename(columns={'Close': 'close'})
df['sma_fast'] = df['close'].rolling(20).mean()
df['sma_slow'] = df['close'].rolling(50).mean()
df.dropna(inplace=True)

df['signal'] = np.where(df['sma_fast'] > df['sma_slow'], 1, -1)
df['ret'] = df['close'].pct_change().fillna(0)
df['strat_ret'] = df['signal'].shift(1).fillna(0) * df['ret']

def perf_stats(returns, freq=252):
    cum_ret = (1 + returns).prod() - 1
    cagr = (1 + cum_ret) ** (freq/len(returns)) - 1
    vol = returns.std() * np.sqrt(freq)
    sharpe = returns.mean() * freq / (returns.std() * np.sqrt(freq)) if vol != 0 else np.nan
    downside = returns[returns < 0].std() * np.sqrt(freq)
    sortino = returns.mean() * freq / downside if downside != 0 else np.nan
    curve = (1 + returns).cumprod()
    peak = curve.cummax()
    dd = (curve/peak - 1)
    max_dd = dd.min()
    return {'CAGR': cagr, 'Sharpe': sharpe, 'Sortino': sortino, 'MaxDD': max_dd}

perf = perf_stats(df['strat_ret'])
perf
